# Cooldown and field rotation on the Triton 200

Resistance of a thin NbSe2 flake through a dilution refrigerator cooldown from
20 K to base, then a 1 T field rotated through 360 degrees in each of three
planes at base.

### Instruments

| Instrument | Role |
| --- | --- |
| Triton 200 | the fridge, thermometers on channels 5, 8 and 13 |
| Mercury iPS | vector magnet, x, y and z |
| Keithley 6221 | sources the current |
| Keithley 2182A | reads the voltage |

### Wiring

`I+ 1, I- 4, V+ 2, V- 3`, and a trigger link cable between the 6221 and the
2182A. Without that cable the 6221 refuses to arm and says so.

The two meters run in pulse delta. The 6221 sends a 12 ms pulse, the 2182A reads
during it, and the current is off in between, which keeps the flake from heating
itself at 20 mK. The pulse reverses each cycle, cancelling thermal EMFs at the
contacts.

### What the data shows

The cooldown crosses the superconducting transition near 7 K, where the
resistance falls to zero.

At base the flake lies in the xy plane. A 1 T field rotated within that plane
stays parallel to the flake, where the upper critical field of a thin
superconductor is large, and the flake stays superconducting all the way round.
In the xz and yz planes the field tilts out of the plane, where the critical
field is far smaller, and the resistance rises toward normal near 90 and 270
degrees.

In [ ]:
### Library Import ###
import csv
import datetime
import os
import pathlib
import time
from collections import deque

import matplotlib.pyplot as plt

from labdrivers.core.errors import InstrumentTimeoutError
from labdrivers.keithley import Keithley2182, Keithley6221
from labdrivers.oxford import MercuryIps, Triton200

## Settings

Everything that changes between runs. The pulse parameters come from the 6221
and 2182A characterization in `R:\Alex\Testing`, where the averaging count was
swept until the noise floor stopped improving.

In [ ]:
### Settings ###
fridge_address = "128.174.165.103"
magnet_address = "192.168.0.10"
source_gpib = 12
meter_gpib = 7

mixing_chamber = 8
still = 5
sample = 13

log_below = 20.0  # K, where logging starts
base = 0.020
stable_readings = 10  # the spread across this window decides settled
stable_spread = 5e-4
cooldown_timeout = 12 * 3600
log_interval = 5.0

pulse_current = 2e-6
pulse_width = 12e-3
pulse_delay = 16e-6
pulse_interval = 5  # power line cycles
pulses_warm = 50  # the signal is noisier warm, so average harder
pulses_cold = 10
warm_above = 15.0
compliance = 10.0

rotation_field = 1.0
rotation_points = 36  # 10 degrees apart
planes = ("xy", "xz", "yz")  # in-plane, then the two out-of-plane
ramp_rate = 0.2  # T/min, per axis

wiring = "I+ 1, I- 4, V+ 2, V- 3, trigger link 6221 to 2182A"
data_directory = pathlib.Path.home() / "measurements"

## Reading a resistance

One pulse train per point. The 6221 has no call that waits for a train to
finish, so the wait is timed from the pulse count and the line frequency. The
spread across the readings becomes the error bar on the point.

In [ ]:
### Auxiliary Functions ###
def add_point(line, axis, x, y):
    """Add one point to a live plot line and rescale."""
    xs, ys = line.get_data()
    line.set_data([*xs, x], [*ys, y])
    axis.relim()
    axis.autoscale_view()
    plt.pause(0.01)


def read_resistance(source, meter, count):
    """Run one pulse-delta train and return the mean resistance and its error."""
    source.clear_buffer()
    source.configure_pulse_delta(
        high=pulse_current,
        low=0.0,
        width=pulse_width,
        measurement_delay=pulse_delay,
        interval=pulse_interval,
        count=count,
    )
    source.arm_pulse_delta()
    source.start()
    time.sleep(count * pulse_interval / meter.line_frequency + 1.0)

    readings = source.read_buffer()
    if len(readings) < count:
        raise RuntimeError(
            f"The pulse train should have given {count} readings, but got "
            f"{len(readings)}."
        )
    mean = sum(readings) / count
    spread = (sum((reading - mean) ** 2 for reading in readings) / count) ** 0.5
    return mean, spread / count**0.5

## The measurement

The cooldown is not stepped. A dilution fridge comes down on its own once
circulation starts, so the loop waits for the mixing chamber to fall below 20 K
and then logs until it is below 20 mK and the last ten readings agree. Averaging
is heavier above 15 K, where the signal is noisier.

The rotation drives both axes of each plane together, so the field turns along
the chord rather than in an L. Every axis returns to zero between planes, so the
third one is never left holding a field from the last.

The shutdowns sit in a `finally`: closing a connection is not the same as
ramping a magnet down or switching a source off, and a `with` block only does
the first.

In [ ]:
### Measurement ###
started = datetime.datetime.now()
data_directory.mkdir(exist_ok=True)
stamp = f"{started:%Y-%m-%d_%H%M}"
header = (
    f"# {started:%Y-%m-%d %H:%M}, wiring {wiring}, pulse delta {pulse_current:.1e} A, "
    f"{pulse_width * 1e3:g} ms every {pulse_interval} PLC\n"
)

# "x" rather than "a", so a rerun cannot quietly append to an old run
cooldown_file = open(data_directory / f"triton_cooldown_{stamp}.csv", "x", newline="")
cooldown_file.write(header)
cooldown = csv.writer(cooldown_file)
cooldown.writerow(
    ["elapsed_s", "still_K", "mixing_chamber_K", "sample_K", "resistance_ohm",
     "error_ohm", "pulses", "in_compliance"]
)

rotation_file = open(data_directory / f"triton_rotation_{stamp}.csv", "x", newline="")
rotation_file.write(header)
rotation_file.write(f"# {rotation_field} T in {rotation_points} steps, {planes}\n")
rotation = csv.writer(rotation_file)
rotation.writerow(
    ["plane", "angle_deg", "field_x_T", "field_y_T", "field_z_T", "mixing_chamber_K",
     "resistance_ohm", "error_ohm", "in_compliance"]
)

plt.ion()
figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))
left.set_xscale("log")
left.set_xlabel("mixing chamber (K)")
left.set_ylabel("R (ohm)")
right.set_xlabel("angle (deg)")
right.set_ylabel("R (ohm)")
right.set_title(f"{rotation_field:g} T rotated in each plane at base")

with Triton200(ip_address=fridge_address) as fridge, MercuryIps(
    ip_address=magnet_address
) as supply, Keithley6221(gpib_address=source_gpib) as source, Keithley2182(
    gpib_address=meter_gpib
) as meter:
    # a wrong address fails here rather than three steps in
    print(source.identify(), meter.identify(), sep="\n")
    print(f"mixing chamber {fridge.temperature(mixing_chamber):.4f} K")

    meter.channel = 1
    meter.integration_time = 5
    meter.set_analog_filter(True, channel=1)
    meter.trigger_source = "external"
    meter.delta = True
    source.compliance = compliance
    source.measurement_unit = "ohms"

    try:
        # Cooldown
        deadline = time.monotonic() + cooldown_timeout
        while fridge.temperature(mixing_chamber) > log_below:
            if time.monotonic() > deadline:
                raise InstrumentTimeoutError(
                    f"The mixing chamber was still above {log_below} K after "
                    f"{cooldown_timeout} s."
                )
            time.sleep(log_interval)

        (line,) = left.plot([], [], ".-")
        recent = deque(maxlen=stable_readings)
        elapsed = time.monotonic()

        while True:
            mixing = fridge.temperature(mixing_chamber)
            pulses = pulses_warm if mixing > warm_above else pulses_cold
            resistance, error = read_resistance(source, meter, pulses)
            cooldown.writerow(
                [time.monotonic() - elapsed, fridge.temperature(still), mixing,
                 fridge.temperature(sample), resistance, error, pulses,
                 int(source.in_compliance())]
            )
            cooldown_file.flush()
            os.fsync(cooldown_file.fileno())
            add_point(line, left, mixing, resistance)

            recent.append(mixing)
            settled = (
                len(recent) == stable_readings
                and max(recent) - min(recent) <= stable_spread
            )
            if mixing < base and settled:
                break
            if time.monotonic() > deadline:
                raise InstrumentTimeoutError(
                    f"The mixing chamber had not settled below {base} K after "
                    f"{cooldown_timeout} s. It reads {mixing} K."
                )
            time.sleep(log_interval)

        # Rotation
        for magnet in supply.magnets.values():
            magnet.field_ramp_rate = ramp_rate

        for plane in planes:
            supply.ramp_all_to_zero()
            (line,) = right.plot([], [], ".-", label=plane)
            right.legend()

            for index, vector in enumerate(
                supply.circle_sweep(rotation_field, rotation_points, plane=plane)
            ):
                angle = 360.0 * index / rotation_points
                for axis, value in vector.items():
                    supply.magnets[axis].field_setpoint = value
                for axis in vector:
                    supply.magnets[axis].ramp_to_setpoint()
                for axis in vector:
                    supply.magnets[axis].wait_for_field()

                field = supply.vector_field()
                resistance, error = read_resistance(source, meter, pulses_cold)
                rotation.writerow(
                    [plane, angle, field["GRPX"], field["GRPY"], field["GRPZ"],
                     fridge.temperature(mixing_chamber), resistance, error,
                     int(source.in_compliance())]
                )
                rotation_file.flush()
                os.fsync(rotation_file.fileno())
                add_point(line, right, angle, resistance)
    finally:
        supply.safe_shutdown()
        source.safe_shutdown()
        fridge.safe_shutdown()
        cooldown_file.close()
        rotation_file.close()
        figure.savefig(data_directory / f"triton_{stamp}.png")

print("Run finished")

## Output

Two CSVs and one PNG in the data directory, all named for the time the run
started. The cooldown file carries every thermometer and the pulse count used
for each point, so a noisy stretch can be traced to the averaging.

![Cooldown on the left, the three rotations on the right](triton_cooldown_and_rotation.png)

The plot above uses made-up data, shaped like a real run.

Both files start with comment lines, so pandas needs `comment="#"`:

```python
import pandas as pd

cooldown = pd.read_csv(cooldown_file.name, comment="#")
rotation = pd.read_csv(rotation_file.name, comment="#")
in_plane = rotation[rotation.plane == "xy"]
```